<div style="font-size:2.3em;font-weight:600;margin-bottom:10px">Agentic Artificial Intelligence</div>
<div style="font-size:1.7em;margin-bottom:14px">Unit 06 · Model Context Protocol</div>

Today our agent gets tools that run in **other programs**.

> **Today's sentence:** MCP is one standard plug between agents and tools. The model still only asks. But now another program runs the tool.

<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px 32px"><div><b>1</b>&nbsp; Opening: one plug for all tools</div><div><b>5</b>&nbsp; An agent with MCP tools</div><div><b>2</b>&nbsp; A server from somebody else</div><div><b>6</b>&nbsp; When to use MCP, and what can go wrong</div><div><b>3</b>&nbsp; Our own server</div><div><b>7</b>&nbsp; Your exercise: a file system server</div><div><b>4</b>&nbsp; Three servers at once</div><div><b>8</b>&nbsp; Wrap-up</div></div>

<div style="height:44px"></div>

## Part 1 of 8 · Opening: one plug for all tools

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What problem does a standard solve?**

#### 1.1 · Let's count

**5** AI apps and **10** services: **5 × 10 = 50** connections.

Every new service needs **5** more.

#### 1.2 · The same idea for AI

<details>
<summary>Click to show</summary>

**MCP** is the **Model Context Protocol**.

A **protocol** is a set of agreed rules for how two programs talk to each other.

> “Think of MCP like a USB-C port for AI applications.” (the official documentation)

Each app and each service build their side only once: **5 + 10 = 15** pieces, not 50.

---

**One sentence for today:** MCP is one standard plug between agents and tools. The model still only asks. But now another program runs the tool.

</details>

#### 1.3 · Server and client

```
┌──────────────┐      MCP      ┌──────────────┐
│    client    │ ◄───────────► │    server    │
│  (our code)  │               │ (a program)  │
└──────────────┘               └──────────────┘
```

**Server:** a program that offers tools and waits for requests.

**Client:** the program that connects to the server and uses its tools. Here: our own code, around our agent.

<div style="height:44px"></div>

## Part 2 of 8 · A server from somebody else

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What do we get from a server that we did not write?**

In [ ]:
# 2.1 · One server, and how to start it
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "langgraph-docs-mcp": {
            "command": "uvx",
            "args": ["--with", "mcp<2", "--from", "mcpdoc", "mcpdoc", "--urls",
                     "LangGraph:https://langchain-ai.github.io/langgraph/llms.txt "
                     "LangChain:https://python.langchain.com/llms.txt"],
            "transport": "stdio"
        }
    }
)

In [ ]:
# 2.2 · What does the server offer?
tools = await client.get_tools()

for i, tool in enumerate(tools, 1):
    print(f"{i}. {tool.name}")
    if tool.description:
        desc = tool.description
        print(f"   {desc}")

    print()

In [ ]:
# 2.3 · The first tool, by hand
print(f'Calling tool {tools[0].name}')
response = await tools[0].ainvoke({'tool_name': 'list_doc_sources'})
print(response[0]["text"])

In [ ]:
# 2.4 · The second tool, by hand
print(f'Calling tool {tools[1].name}')
response = await tools[1].ainvoke({
    'url': 'https://langchain-ai.github.io/langgraph/concepts/server-mcp/'
})
print(response[0]["text"])

#### 2.5 · What does this tell us?

<details>
<summary>Click to show</summary>

A server from somebody else is **not your code**.

You get what its author built in, and you cannot simply change it.

</details>

In [ ]:
# 2.6 · A tool from our package
from agentic_ai.tools.url_fetcher import UrlFetcherTool

response = UrlFetcherTool().to_langchain_tool().invoke({
    'url': 'https://langchain-ai.github.io/langgraph/concepts/server-mcp/',
    'extract_text': True,
})
print(f"Response: {response[:1000]}...")

<div style="height:44px"></div>

## Part 3 of 8 · Our own server

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What is inside an MCP server?**

In [ ]:
# 3.1 · A closer look at this tool
from agentic_ai.tools.url_fetcher import UrlFetcherTool

print("UrlFetcherTool Information:")
print("=" * 60)
fetcher = UrlFetcherTool()
print(f"Name: {fetcher.name}")
print(f"\nDescription:\n{fetcher.description}")

#### 3.2 · The server file: the beginning

From our package, `agentic_ai/mcp/url_fetcher_server.py`:

```python
import asyncio
from typing import Any

from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp.types import Tool, TextContent

from agentic_ai.tools.url_fetcher import UrlFetcherTool

server = Server("url-fetcher-mcp-server")

url_fetcher = UrlFetcherTool()
```

#### 3.3 · The server file: the first job

```python
@server.list_tools()
async def list_tools() -> list[Tool]:
    return [
        Tool(
            name="fetch_url",
            description=url_fetcher.description,
            inputSchema={
                "type": "object",
                "properties": {
                    "url": {
                        "type": "string",
                        "description": "The URL to fetch (must be a valid HTTP/HTTPS URL)"
                    },
                    "extract_text": {
                        "type": "boolean",
                        "description": "Whether to extract text from HTML (default: True). Set to False to get raw HTML.",
                        "default": True
                    }
                },
                "required": ["url"]
            }
        )
    ]
```

#### 3.4 · The server file: the second job

```python
@server.call_tool()
async def call_tool(name: str, arguments: dict[str, Any]) -> list[TextContent]:
    if name == "fetch_url":
        url = arguments.get("url")
        extract_text = arguments.get("extract_text", True)

        if not url:
            return [TextContent(
                type="text",
                text="Error: 'url' parameter is required"
            )]

        try:
            result = url_fetcher._run(url=url, extract_text=extract_text)
            return [TextContent(
                type="text",
                text=result
            )]
        except Exception as e:
            return [TextContent(
                type="text",
                text=f"Error fetching URL: {str(e)}"
            )]
    else:
        return [TextContent(
            type="text",
            text=f"Unknown tool: {name}"
        )]
```

#### 3.5 · The server file: the start

```python
async def main():
    async with stdio_server() as (read_stream, write_stream):
        await server.run(
            read_stream,
            write_stream,
            server.create_initialization_options()
        )


if __name__ == "__main__":
    asyncio.run(main())
```

#### 3.6 · What did we just read?

<details>
<summary>Click to show</summary>

**A server has two jobs:** say which tools you have, and run a tool when asked.

There is **no model** in this file. The model still only asks.

</details>

In [ ]:
# 3.7 · The command that starts it
import sys
import os

mcp_server_path = os.path.join(
    os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd()))),
    "agentic_ai", "mcp", "url_fetcher_server.py"
)

python_path = sys.executable

print(f"Python executable: {python_path}")
print(f"MCP server path: {mcp_server_path}")
print(f"\nTo run the MCP server, use:")
print(f"  {python_path} -m agentic_ai.mcp.url_fetcher_server")

<div style="height:44px"></div>

## Part 4 of 8 · Three servers at once

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What do we get when we connect to three servers?**

#### 4.1 · A shorter way to write a server

**FastMCP:** a normal function, with `@mcp.tool()` above it, much like `@tool` last time.

Our **arXiv** server is written this way: `agentic_ai/mcp/arxiv.py`.

**arXiv** is a free website for research papers.

In [ ]:
# 4.2 · A folder for downloads
from agentic_ai.utils.paths import DATA_DIR

ARXIV_DATA_DIR = (DATA_DIR / "arxiv").as_posix()

In [ ]:
# 4.3 · Three servers
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "langgraph-docs-mcp": {
            "command": "uvx",
            "args": ["--with", "mcp<2", "--from", "mcpdoc", "mcpdoc", "--urls",
                     "LangGraph:https://langchain-ai.github.io/langgraph/llms.txt "
                     "LangChain:https://python.langchain.com/llms.txt"],
            "transport": "stdio"
        },
        "url-fetcher-mcp": {
            "command": sys.executable,
            "args": ["-m", "agentic_ai.mcp.url_fetcher_server"],
            "transport": "stdio"
        },
        "arxiv-server": {
            "command": sys.executable,
            "args": ["-m", "agentic_ai.mcp.arxiv"],
            "env": {"DOWNLOAD_PATH": ARXIV_DATA_DIR},
            "transport": "stdio"
        }
    }
)

In [ ]:
# 4.4 · Ask all three for their tools
tools = await mcp_client.get_tools()

print(f"✅ Successfully connected to URL Fetcher MCP server!")
print(f"Found {len(tools)} tool(s):\n")

for i, tool in enumerate(tools, 1):
    print(f"{i}. {tool.name}")
    if tool.description:
        desc = (tool.description[:200] + "..." if len(tool.description) > 200
                else tool.description)
        print(f"   {desc}\n")
        print(f'    {tool.args}')
    print()

In [ ]:
# 4.5 · Try one: a search on arXiv
articles = await tools[7].ainvoke({'title': 'model context protocol'})

import json
articles = json.loads(articles[0]["text"])
for index, title in enumerate(articles):
    print(f"{index}. {title}")

In [ ]:
# 4.6 · Try another one: fetch_url
print("Testing the fetch_url tool from MCP server:")
print("=" * 60)

test_url = "https://langchain-ai.github.io/langgraph/concepts/server-mcp/"
result = await tools[2].ainvoke({
    "url": test_url,
    "extract_text": True
})

print(f"Fetched content (first 500 chars):\n{result[0]['text'][:500]}...")

<div style="height:44px"></div>

## Part 5 of 8 · An agent with MCP tools

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What does a model do with all these tools?**

In [ ]:
# 5.1 · The start, and a new import
from dotenv import load_dotenv
load_dotenv()

from langgraph.checkpoint.memory import InMemorySaver
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent

model = init_chat_model("google_genai:gemini-3.5-flash-lite")
memory = InMemorySaver()

In [ ]:
# 5.2 · Two small functions
async def build_agent_with_mcp():
    tools = await mcp_client.get_tools()

    agent = create_react_agent(model, tools, checkpointer=memory)
    return agent

async def list_all_tools():
    tools = await mcp_client.get_tools()

    print("Available MCP Tools:")
    print("=" * 60)
    for t in tools:
        print(f"  - {t.name}: {t.description[:80]}...")

In [ ]:
# 5.3 · Build the agent
agent = await build_agent_with_mcp()
config = {"configurable": {"thread_id": "mcp_demo_123"}}

await list_all_tools()

In [ ]:
# 5.4 · Two helpers
async def chat(agent, message):
    print(f"\nQuery: {message}")
    print("=" * 60)

    result = await agent.ainvoke({
        "messages": [
            {"role": "user", "content": message}
        ]
    }, config)

    if result["messages"]:
        print(result["messages"][-1].text)

def get_full_conversation(agent):
    snapshot = agent.get_state(config)
    if snapshot and snapshot.values:
        messages = snapshot.values.get("messages", [])
        print(f"Total messages: {len(messages)}\n")
        for i, msg in enumerate(messages, 1):
            print(f"{i}. {msg.__class__.__name__}:")
            print(f"   {msg.text}\n")
    else:
        print("No messages found in memory for this thread_id")

In [ ]:
# 5.5 · A first question
test_query = "Tell me about langgraph"

await chat(agent, test_query)

In [ ]:
# 5.6 · What happened inside?
import contextlib
import io

with contextlib.redirect_stdout(io.StringIO()) as long_text:
    get_full_conversation(agent)

lines = long_text.getvalue().splitlines()
print(lines[0])
for line, next_line in zip(lines, lines[1:]):
    if line[:1].isdigit() and line.endswith("Message:"):
        print(line, next_line[:70])

In [ ]:
# 5.7 · A second question
test_query = "Is there an article about langgraph?"

await chat(agent, test_query)

In [ ]:
# 5.8 · The next message
test_query = "Please download the first one you mentioned."

await chat(agent, test_query)

In [ ]:
# 5.9 · Look into the folder
from pathlib import Path

for pdf in Path(ARXIV_DATA_DIR).glob("*.pdf"):
    print(pdf.name)

In [ ]:
# 5.10 · Now with a title
test_query = ("Please try again with the title: Agent AI with LangGraph: "
              "A Modular Framework for Enhancing Machine Translation "
              "Using Large Language Models.")

await chat(agent, test_query)

#### 5.11 · What do we take from this?

<details>
<summary>Click to show</summary>

Sometimes the agent does exactly what we asked, and sometimes it does not. Then you test, and you adjust your prompts and your tools.

**One agent, one list of tools, three servers behind it.**

</details>

<div style="height:44px"></div>

## Part 6 of 8 · When to use MCP, and what can go wrong

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**When do we use MCP, and what can go wrong?**

#### 6.1 · Direct or MCP?

| | Direct | MCP |
|:--|:--|:--|
| **How** | your code talks to one service by itself | your code talks to any server in one standard way |
| **Good for** | one or two services, a quick test | many services, tools that other programs should use too |

#### 6.2 · Your keys

**1.** No keys in your code

**2.** Keep them in a `.env` file

**3.** Change your keys from time to time

#### 6.3 · Two rules for servers

<details>
<summary>Click to show</summary>

**1.** Use servers only from people that you trust, and check that they come from the real provider.

**2.** Give a server only the access that it really needs.

---

And remember `uvx` from step 2.1: in one step it downloads a program and runs it.

</details>

<div style="height:44px"></div>

## Part 7 of 8 · Your exercise: a file system server

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#D5DBE3"></span></div>

**What do you build at home?**

#### 7.1 · Your task: a file system server

An MCP server that gives an agent access to files on your computer: `agentic_ai/mcp/filesystem_server.py`.

Its tools: `list_directory`, `read_file`, `search_files`, `get_file_info`.

**Option A:** like `url_fetcher_server.py`, the server that we read today.

**Option B:** `FastMCP`, like `arxiv.py`. It is shorter. The docstring becomes the description.

All details are in your own notebook.

#### 7.2 · Your server can read your own files

**1.** Limit it to one folder with `BASE_PATH`.

**2.** Block `..` in a path. Two dots mean “one folder up”.

**3.** Block hidden files such as `.env`. Your API keys are in there.

#### 7.3 · When you test it

```bash
python -m agentic_ai.mcp.filesystem_server
```

**Nothing seems to happen, and that is correct.** The server waits for messages.

Stop it with Control C. A long error text with `KeyboardInterrupt` is normal too.

Do not use `print` in a `stdio` server.

#### 7.4 · Your agent, and an optional part

MCP tools are async, so you need `await` and `ainvoke`.

**Use:** `create_react_agent`, as today, or `ToolAgent` from `agentic_ai.agents.tool_agent_async`, with `arun`.

**Not:** our normal `ToolAgent`. It gives a `NotImplementedError`.

---

**Optional, more tools:** `write_file`, `create_directory`, `find_in_files`, `get_directory_tree`. If a tool can write, think about the checks first.

<div style="height:44px"></div>

## Part 8 of 8 · Wrap-up

<div style="display:flex;gap:6px;margin:0 0 14px 0"><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span><span style="flex:1;height:10px;border-radius:5px;background:#1F3A5F"></span></div>

**What should you remember from today?**

#### 8.1 · Five things

**1.** MCP is one standard plug between agents and tools.

**2.** A server has two jobs: say which tools you have, and run a tool when asked.

**3.** With `stdio`, the client starts the server as another program, and they send text to each other.

**4.** MCP tools are async, so we use `await` and `ainvoke`.

**5.** A server from somebody else is somebody else's code, and its description is somebody else's prompt.

---

**The model still only asks.**

**At home:** the file system server.

**Next time, Unit 7:** we look inside our agents with **tracing**.